In [9]:
import os.path
import csv
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

import clustbench
import genieclust
from sklearn.cluster import (
    KMeans, AgglomerativeClustering, SpectralClustering,
    Birch, MiniBatchKMeans, SpectralCoclustering
)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.decomposition import (
    LatentDirichletAllocation
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer, CountVectorizer, HashingVectorizer
)

from sklearn.metrics import (
    accuracy_score, rand_score, adjusted_rand_score,
    fowlkes_mallows_score, mutual_info_score, adjusted_mutual_info_score, normalized_mutual_info_score
)
import hdbscan
from kmodes.kmodes import KModes
# from kmedoids import KMedoids
# from sklearn_extra.cluster import KMedoids
from sklearn import metrics
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModel
)
import umap
import torch

In [2]:
# model_id = "sentence-transformers/gemini-embedding-exp-03-07"
model_id = "Salesforce/SFR-Embedding-2_R"
hf_token = ""

import requests
from retry import retry

api_url = f"https://api-inference.huggingface.co/pipeline/feature-extraction/{model_id}"
headers = {"Authorization": f"Bearer {hf_token}"}

@retry(tries=3, delay=10)
def query(texts):
    response = requests.post(api_url, headers=headers, json={"inputs": texts})
    result = response.json()
    if isinstance(result, list):
      return result
    elif list(result.keys())[0] == "error":
      raise RuntimeError(
          "The model is currently loading, please re-run the query."
          )
    
texts = ["How do I get a replacement Medicare card?",
        "What is the monthly premium for Medicare Part B?",
        "How do I terminate my Medicare Part B (medical insurance)?",
        "How do I sign up for Medicare?",
        "Can I sign up for Medicare Part B if I am working and have health insurance through an employer?",
        "How do I sign up for Medicare Part B if I already have Part A?",
        "What are Medicare late enrollment penalties?",
        "What is Medicare and who can get it?",
        "How can I get help with my Medicare Part A and Part B premiums?",
        "What are the different parts of Medicare?",
        "Will my Medicare premiums be higher because of my higher income?",
        "What is TRICARE ?",
        "Should I sign up for Medicare Part B if I have Veterans’ Benefits?"]

output = query(texts)

RuntimeError: The model is currently loading, please re-run the query.

In [5]:
import pandas as pd

embeddings = pd.DataFrame(output)
print(embeddings)

         0         1         2         3         4         5         6    \
0  -0.023889  0.055259 -0.011655 -0.033414 -0.012260 -0.024873 -0.012663   
1  -0.012688  0.046874 -0.010502 -0.020384 -0.013361  0.042322  0.016628   
2   0.000494  0.119412  0.005229 -0.092734  0.007773 -0.005325  0.034506   
3  -0.029711  0.023298 -0.057041 -0.012183 -0.013710  0.029796  0.063739   
4  -0.025628  0.070389 -0.017380 -0.056567  0.028577  0.052822  0.067062   
5  -0.022656  0.021160  0.005105 -0.046494  0.009074  0.041495  0.054268   
6  -0.002911  0.060791 -0.009176 -0.006133  0.040492  0.036594  0.002055   
7  -0.080526  0.059888 -0.048847 -0.040176 -0.063342  0.041848  0.119045   
8  -0.034388  0.072501  0.014440 -0.036695  0.014019  0.063070  0.034682   
9  -0.005964  0.025044 -0.003182 -0.025243 -0.039823 -0.012772  0.044713   
10 -0.039008 -0.010609 -0.007383 -0.050190 -0.002518 -0.041641  0.026969   
11 -0.095983 -0.063012 -0.116906 -0.059075 -0.051323 -0.003439  0.018687   
12 -0.011629

In [11]:
import umap

In [10]:
def preprocess_data(X):
    """Convert each row of X into a string, joined by spaces."""
    return [" ".join(map(str, row)) for row in X]

def generate_local_UMAP_embedding(X):
    return umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric='euclidean'
        # random_state=42
    ).fit_transform(X)

def generate_global_UMAP_embedding(X):
    return umap.UMAP(
        n_components=2,
        n_neighbors=35,
        min_dist=0.3,
        metric='euclidean'
        # random_state=42
    ).fit_transform(X)

# -------------------------------------------------------------------
# 2. Text-based Embedding Functions
# -------------------------------------------------------------------
def generate_TFIDF_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    vectorizer = TfidfVectorizer(max_features=500)
    res = vectorizer.fit_transform(X_as_str).toarray()
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

# DOES NOT EXIST ON HUGGINGFACE
# def generate_gemini_embedding(X, X_as_str=None):
#     if X_as_str is None:
#         X_as_str = preprocess_data(X)
#     model = SentenceTransformer('google/gemini-embedding-exp-03-07')
#     res = model.encode(X_as_str, show_progress_bar=False)
#     return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_gritlm_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    model = SentenceTransformer('GritLM/GritLM-7B')
    res = model.encode(X_as_str, show_progress_bar=False)
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_sfr_mistral_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    model = SentenceTransformer('Salesforce/SFR-Embedding-Mistral')
    res = model.encode(X_as_str, show_progress_bar=False)
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_sfr_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    sf_model = SentenceTransformer('Salesforce/SFR-Embedding-2_R')
    res = sf_model.encode(X_as_str, show_progress_bar=False)
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_DistilBERT_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained("distilbert-base-uncased")

    def get_distilbert_embedding(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        # Mean pooling over token embeddings
        return outputs.last_hidden_state.mean(dim=1).detach().numpy()[0]

    res = np.array([get_distilbert_embedding(txt) for txt in X_as_str])
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_Doc2Vec_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    documents = [TaggedDocument(words=txt.split(), tags=[str(i)]) for i, txt in enumerate(X_as_str)]
    doc2vec_model = Doc2Vec(vector_size=50, min_count=1, epochs=40)
    doc2vec_model.build_vocab(documents)
    doc2vec_model.train(documents, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
    res = np.array([doc2vec_model.infer_vector(txt.split()) for txt in X_as_str])
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]

def generate_multilingual_e5_large_instruct_embedding(X, X_as_str=None):
    if X_as_str is None:
        X_as_str = preprocess_data(X)
    model = SentenceTransformer('intfloat/multilingual-e5-large-instruct')
    res = model.encode(X_as_str, show_progress_bar=False)
    return [generate_local_UMAP_embedding(res), generate_global_UMAP_embedding(res)]



In [11]:
def generate_embeddings(X):
    # Precompute the text representation once
    X_as_str = preprocess_data(X)
    
    # List your embedding functions as tuples: (name, function, is_text_based)
    embedding_functions = [
        ("UMAP_local", generate_local_UMAP_embedding, False),
        ("UMAP_global", generate_global_UMAP_embedding, False),
        # ("TFIDF", generate_TFIDF_embedding, True),
        # ("DistilBERT", generate_DistilBERT_embedding, True),
        # ("Doc2Vec", generate_Doc2Vec_embedding, True),
        # ("multilingual_e5_large_instruct", generate_multilingual_e5_large_instruct_embedding, True),
        # ("sfr_2r", generate_sfr_embedding, True),
        # ("sfr_mistral", generate_sfr_mistral_embedding, True),
        # ("gritlm", generate_gritlm_embedding, True),
    ]
    
    results_dict = {}
    with ThreadPoolExecutor() as executor:
        futures = {}
        for name, func, is_text in embedding_functions:
            # For text-based functions, pass the precomputed X_as_str; otherwise, just pass X.
            if is_text:
                futures[executor.submit(func, X, X_as_str)] = name
            else:
                futures[executor.submit(func, X)] = name

        for future in futures:
            func_name = futures[future]
            try:
                result = future.result()
            except Exception as e:
                print(f"Error in {func_name}: {e}")
                results_dict[func_name] = None
                continue
            
            # If the function returned a list of length 2, assume [local, global] structure.
            if isinstance(result, list) and len(result) == 2:
                results_dict[func_name + "_local"] = result[0]
                results_dict[func_name + "_global"] = result[1]
            else:
                # Otherwise, store under the original name
                results_dict[func_name] = result

    results_dict['Base'] = X
    return results_dict


In [20]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

import pickle

def get_cached_embeddings(collection, dataset, X):
    cache_dir = "embedding_cache_ay"
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"{collection}_{dataset}_embeddings.pkl")
    
    if os.path.exists(cache_file):
        print("Loading embeddings from cache:", cache_file)
        with open(cache_file, "rb") as f:
            embeddings = pickle.load(f)
    else:
        print("Cache not found. Generating embeddings...")
        embeddings = generate_embeddings(X)
        with open(cache_file, "wb") as f:
            pickle.dump(embeddings, f)
        print("Embeddings cached at:", cache_file)
    return embeddings

def load_data(collection, dataset):
    benchmark = clustbench.load_dataset(collection, dataset, url=data_url)
    X = benchmark.data
    print("Loaded: ", X.shape[0], " | Dimension: ", X.shape[1], " | Label count: ", len(benchmark.labels))
    print("Getting embeddings (with caching)...")
    X_embedded_dict = get_cached_embeddings(collection, dataset, X)
    return X, benchmark, X_embedded_dict

In [21]:
"""
each dataset can have multiple labels, 
pick one at a time and that defines your partition size, aka k

Overall, Genie returned a clustering quite similar to the reference one. We may consider 107
(namely, c11 + c22 + c33 ) out of the 120 input points as correctly grouped. In particular, 
all the red and green reference points (the 2nd and the 3rd row) have been properly discovered.

Normalized Clustering Accuracy (NCA) 
NCA is the averaged percentage of correctly classified points in each cluster 
above the perfectly uniform label distribution.
            
"""

from hdbscan import flat
def predict(embedding_technique, X, label, benchmark, clustering_method, plot=False):

    y_true = benchmark.labels[label] 
    (k := max(y_true))  # or benchmark.n_clusters[0]
    m = max(min(y_true),2)
    method = clustering_method.lower()

    # Define the clustering model
    if method == "genie":
        model = genieclust.Genie(n_clusters=k)  # using default parameters
    elif method == "kmeans":
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
    elif method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=k)
    elif method == "spectral":
        model = SpectralClustering(n_clusters=k, random_state=42)
    elif method == "gaussianmixture":
        model = GaussianMixture(n_components=k, random_state=42)
    elif method == "hdbscan":
        # model = hdbscan.HDBSCAN(min_cluster_size=m)
        model = flat.HDBSCAN_flat(X, k, prediction_data=True)
    elif method == "kmodes":
        model = KModes(n_clusters=k, random_state=42, init="Huang")
    elif method == "birch":
        model = Birch(n_clusters=k)
    elif method == "minibatchkmeans":
        model = MiniBatchKMeans(n_clusters=k, random_state=42)
    # elif method == "kmedoids":
    #     model = KMedoids(n_clusters=k, random_state=42)
    elif method == "latentdirichletallocation":
        X = np.maximum(X, 0)
        model = LatentDirichletAllocation(n_components=k, random_state=42)
    elif method == "spectralcoclustering":
        model =  SpectralCoclustering(n_clusters=k)
    elif method == "bayesiangaussianmixture":
        model = BayesianGaussianMixture(n_components=k)   

    print("The model: " +  method + " has been trained now getting y_pred") 
   
    # Fit the model and predict the cluster labels
    if method == "gaussianmixture":  # Gaussian uses predict instead of fit_predict
        (y_pred := model.fit(X).predict(X) + 1)
    elif method == "latentdirichletallocation":
        model.fit(X)
        y_pred = model.transform(X).argmax(axis=1) + 1
    elif method == "spectralcoclustering":
        model.fit(X)
        y_pred = y_pred = model.row_labels_ + 1
    # elif method == "optics" or method == "hdbscan" or method == "dbscan":
    elif method == "hdbscan":
        y_pred, _ = flat.approximate_predict_flat(model, X, k)
        # y_pred = model.fit_predict(X)
        # unique_labels = np.unique(y_pred)
        # if -1 in unique_labels:
        #     y_pred = np.where(y_pred == -1, max(unique_labels) + 1, y_pred)  # Assign noise to a new cluster
        # y_pred += 1
    else:
        (y_pred := model.fit_predict(X) + 1)
        
    assert len(y_true) == len(y_pred), "Length of y_true: " + str(len(y_true)) +  "and y_pred: " + str(len(y_pred)) + "are not the same"
    print("y_true dimensions: " + str(y_true.shape))
    print("y_pred dimensions: " + str(y_pred.shape))
    print("max(y_true): " + str(max(y_true)))
    print("max(y_pred): " + str(max(y_pred)))
    print("min(y_true): " + str(min(y_true)))
    print("min(y_pred): " + str(min(y_pred)))

    nca = clustbench.get_score(y_true, y_pred)
    cf = metrics.confusion_matrix(y_true, y_pred)
    r = rand_score(y_true, y_pred)
    ar = adjusted_rand_score(y_true, y_pred)
    fm = fowlkes_mallows_score(y_true, y_pred)
    mi = mutual_info_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    a = accuracy_score(y_true, y_pred)
        


    if plot:
        plt.subplot(1, 2, 1)
        model.plots.plot_scatter(X, labels=y_true-1, axis="equal", title="y_true")
        plt.subplot(1, 2, 2)
        model.plots.plot_scatter(X, labels=y_pred-1, axis="equal", title="y_pred")
        plt.show()

    return cf, nca, r, ar, fm, mi, nmi, ami, a
    
    

In [12]:
import os

print(os.getcwd()) # run to check current working directory and update file path if needed

/Users/aaditya/development/embedding_based_clustering_research/framework


In [14]:
eval_collections = {
    "wut": [
        "x1",
        "x2",
        "x3",
        "z1",
        "z2",
        "z3",
        "circles",
        "cross",
        "graph",
        "isolation",
        "labirynth",
        "mk1",
        "mk2",
        "mk3",
        "mk4",
        "olympic",
        "smile",
        "stripes",
        "trajectories",
        "trapped_lovers",
        "twosplashes",
        "windows"
    ],
    "other": [
        "chameleon_t4_8k",
        "chameleon_t5_8k",
        "chameleon_t7_10k",
        "chameleon_t8_8k",
        "hdbscan",
        "iris",
        "iris5",
        "square"
    ],
    "graves": [
        "dense",
        "fuzzyx",
        "line",
        "parabolic",
        "ring",
        "ring_noisy",
        "ring_outliers",
        "zigzag",
        "zigzag_noisy",
        "zigzag_outliers"
    ],
    "sipu": [
        "a1",
        "a2",
        "a3",
        "aggregation",
        "birch1",
        "birch2",
        "compound",
        "d31",
        "flame",
        "jain",
        "pathbased",
        "r15",
        "s1",
        "s2",
        "s3",
        "s4",
        "spiral",
        "unbalance",
        "worms_2",
        "worms_64"
    ],
    "fcps": [
        "atom",
        "chainlink",
        "engytime",
        "hepta",
        "lsun",
        "target",
        "tetra",
        "twodiamonds",
        "wingnut"
    ],
}
# REMEMBER YOU REMOVED KMEDOIDS FROM THE LIST OF CLUSTERING METHODS
clustering_methods = ["genie", "kmeans", "agglomerative", "spectral","gaussianmixture", "hdbscan", "kmodes", "birch", "minibatchkmeans" , "latentdirichletallocation", "spectralcoclustering", "bayesiangaussianmixture"]
result_csv = "results/aaditya.csv"

In [15]:
""" 
Run to set the column names for the csv file
"""
import os
import csv

if os.path.exists(result_csv):
    print("File already exists")
else:
    try:
        with open(result_csv, mode='w', newline='') as file: 
            writer = csv.writer(file)
            writer.writerow([
                "Collection", "Dataset", "Clustering Method", "Label", "Embedding",
                "Original Dimensions", "Embedding Dimensions",
                "CF", "NCA Score", "R", "AR", "FM", "MI", "NMI", "AMI", "A"
            ])
    except Exception as e:
        print("Error writing to file: ", e)



File already exists


In [9]:
# TODO: maybe create a cache or temporary storage for the embeddings
# TODO: parallelize the embedding and clustering process per dataset? 

In [19]:
with open(result_csv, mode='a', newline='') as file:
    writer = csv.writer(file)
    for collection, datasets in eval_collections.items():
        for dataset in datasets:
            print(f"Collection: {collection}, Dataset: {dataset}")
            X, benchmark, X_embedded_dict = load_data(collection, dataset)
            
            # Get the original data dimensions (e.g., "150 x 4")
            orig_dim = f"{X.shape[0]} x {X.shape[1]}"
            
            for label in range(0, len(benchmark.labels)):
                for embedding_technique, embedded_data in X_embedded_dict.items():
                    # Determine the embedding dimensions if available
                    if hasattr(embedded_data, "shape"):
                        embed_dim = f"{embedded_data.shape[0]} x {embedded_data.shape[1]}"
                    else:
                        embed_dim = "Unknown"

                    print(f"Embedding technique Used: {embedding_technique} for Collection: {collection}, Dataset: {dataset}")
                    
                    for clustering_method in clustering_methods:
                        cf, nca_score, r, ar, fm, mi, nmi, ami, a = predict(
                            embedding_technique, 
                            embedded_data, 
                            label, 
                            benchmark, 
                            clustering_method
                        )
                        print(cf)
                        cf_str = ", ".join(map(str, cf.flatten())) if hasattr(cf, "flatten") else ", ".join(map(str, cf))
    
                        writer.writerow([
                            collection,          # Collection
                            dataset,             # Dataset
                            clustering_method,   # Clustering Method
                            label,               # Label index
                            embedding_technique, # Embedding technique used
                            orig_dim,            # Original data dimensions
                            embed_dim,           # Embedding dimensions
                            f"cf: {cf} ",        # Confusion matrix
                            f"nca: {nca_score} ",# Normalized Clustering Accuracy
                            f"r: {r} ",         # Rand index
                            f"ar: {ar} ",       # Adjusted Rand index
                            f"fm: {fm} ",       # Fowlkes-Mallows index
                            f"mi: {mi} ",       # Mutual Information
                            f"nmi: {nmi} ",     # Normalized Mutual Information
                            f"ami: {ami} ",     # Adjusted Mutual Information
                            f"a: {a} "         # Accuracy Score
                        ])


Collection: wut, Dataset: x1
Loaded:  120  | Dimension:  2  | Label count:  1
Getting embeddings (with caching)...
Loading embeddings from cache: embedding_cache_ay/wut_x1_embeddings.pkl
Embedding technique Used: UMAP_local for Collection: wut, Dataset: x1
The model: genie has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [40  0  0]
 [ 0  0 30]]
The model: kmeans has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[50  0  0]
 [ 0  0 40]
 [ 0 30  0]]
The model: agglomerative has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[50  0  0]
 [ 0  0 40]
 [ 0 30  0]]
The model: spectral has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y

/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/clustbench/score.py:91: UserWarning: `results[3]` is not available.
  warnings.warn("`results[%d]` is not available." % k)
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/clustbench/score.py:102: RuntimeWarning: All-NaN axis encountered
  return np.nanmax(scores)


y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0  0 50]
 [40  0  0]
 [ 0 30  0]]
The model: latentdirichletallocation has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 2
[[ 0  0 50]
 [ 0  0 40]
 [ 0 30  0]]
The model: spectralcoclustering has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [ 0  0 40]
 [30  0  0]]
The model: bayesiangaussianmixture has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[50  0  0]
 [ 0  0 40]
 [ 0 30  0]]
Embedding technique Used: UMAP_global for Collection: wut, Dataset: x1
The model: genie has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
m

/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/clustbench/score.py:91: UserWarning: `results[3]` is not available.
  warnings.warn("`results[%d]` is not available." % k)
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/clustbench/score.py:102: RuntimeWarning: All-NaN axis encountered
  return np.nanmax(scores)


y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [ 4  0 36]
 [ 0  0 30]]
The model: spectralcoclustering has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [40  0  0]
 [ 0  0 30]]
The model: bayesiangaussianmixture has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [40  0  0]
 [ 0  0 30]]
Embedding technique Used: Base for Collection: wut, Dataset: x1
The model: genie has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true): 1
min(y_pred): 1
[[ 0 50  0]
 [40  0  0]
 [ 0  0 30]]
The model: kmeans has been trained now getting y_pred
y_true dimensions: (120,)
y_pred dimensions: (120,)
max(y_true): 3
max(y_pred): 3
min(y_true):

...

y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[399   1   0   0]
 [298   0   1   1]
 [200   0   0   0]
 [100   0   0   0]]
The model: birch has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[346  54   0   0]
 [  0   0   0 300]
 [  0   0 200   0]
 [  0 100   0   0]]
The model: minibatchkmeans has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[  0  58   0 342]
 [  0   0 300   0]
 [200   0   0   0]
 [  0 100   0   0]]
The model: latentdirichletallocation has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[  0  24   0 376]
 [  1   0 299   0]
 [200   0   0   0]
 [  0  76   0  24]]
The model: spectralcoclustering has been traine

/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/hdbscan/flat.py:155: UserWarning: Cannot predict more than 3 with cluster selection method 'eom'. Changing to method 'leaf'...
  warn(f"Cannot predict more than {max_eom_clusters} with cluster "
/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/clustbench/score.py:91: UserWarning: `results[4]` is not available.
  warnings.warn("`results[%d]` is not available." % k)
/Users/aaditya/miniforge3/envs/embedding_research/lib/py

y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[399   0   0   1]
 [298   1   1   0]
 [200   0   0   0]
 [100   0   0   0]]
The model: birch has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[398   2   0   0]
 [  0 300   0   0]
 [  0   0 200   0]
 [  4   0   1  95]]
The model: minibatchkmeans has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[  0 400   0   0]
 [  0   0 300   0]
 [200   0   0   0]
 [  0   0   0 100]]
The model: latentdirichletallocation has been trained now getting y_pred
y_true dimensions: (1000,)
y_pred dimensions: (1000,)
max(y_true): 4
max(y_pred): 4
min(y_true): 1
min(y_pred): 1
[[279   0 121   0]
 [ 28   0 246  26]
 [  0   0   0 200]
 [ 31   0   0  69]]
The model: spectralcoclustering has been traine

/Users/aaditya/miniforge3/envs/embedding_research/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import pandas as pd

def filter_and_compare_csv(file_path):
    # Define column names based on the CSV structure
    col_names = [
        "Collection", "Dataset", "Clustering Method", "Label", "Embedding", "Original Dimensions", "Embedding Dimensions",
        "CF", "NCA", "r", "ar", "fm", "mi", "nmi", "ami", "a"
    ]
    
    # Read CSV without a header, assigning our own column names
    df = pd.read_csv(file_path, header=None, names=col_names)
    
    # Remove extra whitespace from all string cells
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    
    # Function to clean a numeric field with a prefix
    def clean_numeric(value, prefix):
        if isinstance(value, str):
            value = value.replace(prefix, "").strip()
        try:
            return float(value)
        except Exception:
            return None

    # Clean the NCA column by removing the "nca:" prefix and converting to float
    df['NCA'] = df['NCA'].apply(lambda x: clean_numeric(x, 'nca:'))
    
    # Group by the specified columns
    grouped = df.groupby(["Collection", "Dataset", "Clustering Method", "Label"])
    
    filtered_rows = []
    
    # Iterate over each group
    for name, group in grouped:
        # Find the "Base" row in the Embedding column
        base_row = group[group['Embedding'] == 'Base']
        if not base_row.empty:
            base_value = base_row.iloc[0]['NCA']
            base_row_list = base_row.iloc[0].tolist()
            base_added = False
            
            # Compare each row's NCA value to the base_value
            for index, row in group.iterrows():
                if row['Embedding'] != 'Base' and row['NCA'] > base_value:
                    if not base_added:
                        filtered_rows.append(base_row_list)
                        base_added = True
                    filtered_rows.append(row.tolist())
    
    # Create a new DataFrame from the filtered rows and remove duplicates
    filtered_df = pd.DataFrame(filtered_rows, columns=df.columns)
    filtered_df = filtered_df.drop_duplicates()
    
    return filtered_df

# Example usage
file_path = "/Users/cajoshuapark/Dev/research/embedding_based_clustering_research/framework/results/josh.csv"
filtered_df = filter_and_compare_csv(file_path)

# Select only the desired columns for display
display_columns = ["Collection", "Dataset", "Clustering Method", "Label", "NCA"]
print(filtered_df[display_columns])
